In [13]:
# Imports & Data Loading

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

In [14]:
# Load Data & Recreate Features / Labels
# Load dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame

feature_cols = ["MedInc", "AveRooms", "AveOccup", "HouseAge"]
target_reg = "MedHouseVal"

X = df[feature_cols].values
y_reg = df[target_reg].values

# Binary label for "high-value / high-rent-risk"
threshold = np.median(y_reg)
y_cls = (y_reg > threshold).astype(int)

print("Threshold (median house value):", threshold)
print("Label distribution:", np.bincount(y_cls))

Threshold (median house value): 1.797
Label distribution: [10323 10317]


In [15]:
# Train/Test Split & Normalization
X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (16512, 4)
Test shape: (4128, 4)


In [16]:
def normalize_features(X):
    mu = np.mean(X, axis=0)
    ptp = np.ptp(X, axis=0)
    X_norm = (X - mu) / ptp
    return X_norm, mu, ptp

def apply_normalization(X, mu, ptp):
    return (X - mu) / ptp

X_train_norm, mu, ptp = normalize_features(X_train)
X_test_norm = apply_normalization(X_test, mu, ptp)


In [17]:
# Reuse Linear & Logistic Models
# Linear regression utilities
def compute_cost_linear(X, y, w, b):
    m = X.shape[0]
    preds = X @ w + b
    error = preds - y
    cost = (1 / (2 * m)) * np.sum(error ** 2)
    return cost

def gradient_linear(X, y, w, b):
    m = X.shape[0]
    preds = X @ w + b
    error = preds - y
    dj_dw = (1 / m) * (X.T @ error)
    dj_db = (1 / m) * np.sum(error)
    return dj_dw, dj_db

def gradient_descent_linear(X, y, w_in, b_in, alpha, num_iters):
    w = w_in.copy()
    b = float(b_in)
    J_history = []

    for i in range(num_iters):
        dj_dw, dj_db = gradient_linear(X, y, w, b)
        w -= alpha * dj_dw
        b -= alpha * dj_db
        J_history.append(compute_cost_linear(X, y, w, b))

    return w, b, J_history

def predict_linear(X, w, b):
    return X @ w + b


In [18]:
# Logistic regression utilities
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_cost_logistic(X, y, w, b):
    m = X.shape[0]
    z = X @ w + b
    f_wb = sigmoid(z)
    eps = 1e-8
    cost = -(1/m) * np.sum(y * np.log(f_wb + eps) + (1 - y) * np.log(1 - f_wb + eps))
    return cost

def gradient_logistic(X, y, w, b):
    m = X.shape[0]
    z = X @ w + b
    f_wb = sigmoid(z)
    error = f_wb - y
    dj_dw = (1/m) * (X.T @ error)
    dj_db = (1/m) * np.sum(error)
    return dj_dw, dj_db

def gradient_descent_logistic(X, y, w_in, b_in, alpha, num_iters):
    w = w_in.copy()
    b = float(b_in)
    J_history = []

    for i in range(num_iters):
        dj_dw, dj_db = gradient_logistic(X, y, w, b)
        w -= alpha * dj_dw
        b -= alpha * dj_db
        J_history.append(compute_cost_logistic(X, y, w, b))

    return w, b, J_history

def predict_proba(X, w, b):
    return sigmoid(X @ w + b)

def predict_class(X, w, b, threshold=0.5):
    proba = predict_proba(X, w, b)
    return (proba >= threshold).astype(int)


In [19]:
# Train Both Models
# Train linear regression on normalized features
m, n = X_train_norm.shape
w_lin_init = np.zeros(n)
b_lin_init = 0.0

alpha_lin = 0.1
num_iters_lin = 500

w_lin, b_lin, J_hist_lin = gradient_descent_linear(
    X_train_norm, y_reg_train, w_lin_init, b_lin_init, alpha_lin, num_iters_lin
)

In [20]:
# Train logistic regression on same normalized features
w_log_init = np.zeros(n)
b_log_init = 0.0

alpha_log = 0.1
num_iters_log = 500

w_log, b_log, J_hist_log = gradient_descent_logistic(
    X_train_norm, y_cls_train, w_log_init, b_log_init, alpha_log, num_iters_log
)

In [21]:
# Build the Explanation Prompt (LLM-Friendly Text)
feature_cols


['MedInc', 'AveRooms', 'AveOccup', 'HouseAge']

In [22]:
def build_explanation_prompt(features_dict, price_pred, risk_class, risk_proba):
    """
    Build a natural-language prompt for an LLM explaining the prediction.
    """
    lines = [
        "You are an AI assistant explaining a housing price and rent-risk prediction to a non-technical user.",
        "",
        "Here are the house features:",
    ]
    for k, v in features_dict.items():
        lines.append(f"- {k}: {v}")
    lines += [
        "",
        f"The machine learning model predicted:",
        f"- Estimated house value: {price_pred:.2f} (in 100k USD units)",  # adjust description as needed
        f"- Rent/price risk class: {risk_class} (0 = lower, 1 = higher)",
        f"- Probability of higher risk: {risk_proba:.2f}",
        "",
        "Explain in simple, friendly language why the model might make this prediction,",
        "what factors are most important, and how the user should interpret it.",
    ]
    return "\n".join(lines)


In [23]:
# Stub LLM Function (Safe to Run Without an API Key)
def explain_with_llm_stub(prompt: str) -> str:
    """
    Placeholder: in a real system, this would call an LLM (OpenAI, etc.).
    For now, just return the prompt so the flow is visible.
    """
    return "=== LLM EXPLANATION (stub) ===\n\n" + prompt


In [24]:
# (Optional) Real LLM Integration Sketch
# %pip uninstall -y openai
# %pip install -U openai

import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def explain_with_llm_openai(prompt: str) -> str:
    resp = client.chat.completions.create(
        model="gpt-4.1-mini",  # keep your model name (change if your account uses a different one)
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4,
    )
    return resp.choices[0].message.content


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
# Pick Some Test Examples and Generate Explanations
# Pick a few indices from the test set
num_examples = 5
indices = np.random.choice(X_test_norm.shape[0], size=num_examples, replace=False)

for idx in indices:
    x_raw = X_test[idx]           # original scale
    x_norm = X_test_norm[idx:idx+1]  # keep 2D for @

    # Predictions
    price_pred = predict_linear(x_norm, w_lin, b_lin)[0]
    risk_proba = predict_proba(x_norm, w_log, b_log)[0]
    risk_class = int(risk_proba >= 0.5)

    # Build features dict (pretty printing)
    features_dict = {name: float(val) for name, val in zip(feature_cols, x_raw)}

    prompt = build_explanation_prompt(features_dict, price_pred, risk_class, risk_proba)
    explanation = explain_with_llm_stub(prompt)
    # explanation = explain_with_llm_openai(prompt)


    print("========================================")
    print("RAW FEATURES:", features_dict)
    print(f"PREDICTED PRICE   : {price_pred:.3f}")
    print(f"PREDICTED RISK    : {risk_class} (P(high risk) = {risk_proba:.3f})")
    print("----------------------------------------")
    print(explanation)
    print("\n\n")


RAW FEATURES: {'MedInc': 4.5368, 'AveRooms': 5.720250521920668, 'AveOccup': 3.0647181628392484, 'HouseAge': 37.0}
PREDICTED PRICE   : 2.337
PREDICTED RISK    : 1 (P(high risk) = 0.532)
----------------------------------------
=== LLM EXPLANATION (stub) ===

You are an AI assistant explaining a housing price and rent-risk prediction to a non-technical user.

Here are the house features:
- MedInc: 4.5368
- AveRooms: 5.720250521920668
- AveOccup: 3.0647181628392484
- HouseAge: 37.0

The machine learning model predicted:
- Estimated house value: 2.34 (in 100k USD units)
- Rent/price risk class: 1 (0 = lower, 1 = higher)
- Probability of higher risk: 0.53

Explain in simple, friendly language why the model might make this prediction,
what factors are most important, and how the user should interpret it.



RAW FEATURES: {'MedInc': 3.2705, 'AveRooms': 4.772479564032698, 'AveOccup': 2.0490463215258856, 'HouseAge': 52.0}
PREDICTED PRICE   : 2.218
PREDICTED RISK    : 1 (P(high risk) = 0.523)
--